In [2]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Notebook is working!")

Pandas version: 2.3.3
NumPy version: 2.2.6
Notebook is working!


In [3]:
import pandas as pd

basics_path = "../data/raw/title.basics.tsv"
ratings_path = "../data/raw/title.ratings.tsv"

basic_columns = [
    "tconst",
    "titleType",
    "primaryTitle",
    "startYear",
    "runtimeMinutes",
    "genres"
]

rating_columns = [
    "tconst",
    "averageRating",
    "numVotes"
]

basics = pd.read_csv(
    basics_path,
    sep="\t",
    na_values="\\N",
    usecols=basic_columns,
    low_memory=False
)

ratings = pd.read_csv(
    ratings_path,
    sep="\t",
    na_values="\\N",
    usecols=rating_columns
)

print("Basics shape:", basics.shape)
print("Ratings shape:", ratings.shape)

Basics shape: (12732739, 6)
Ratings shape: (1707914, 3)


In [3]:
basics.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,1894.0,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,1892.0,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,1892.0,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,1892.0,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,1893.0,1,Short


In [4]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2226
1,tt0000002,5.4,324
2,tt0000003,6.4,2372
3,tt0000004,5.0,201
4,tt0000005,6.2,3087


In [5]:
basics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12732739 entries, 0 to 12732738
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   tconst          object 
 1   titleType       object 
 2   primaryTitle    object 
 3   startYear       float64
 4   runtimeMinutes  object 
 5   genres          object 
dtypes: float64(1), object(5)
memory usage: 582.9+ MB


In [6]:
movies = basics[basics["titleType"] == "movie"].copy()

print("Total movie records:", len(movies))

Total movie records: 754810


In [7]:
movies.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,1894.0,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897.0,100,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,1905.0,100,NaN
570,tt0000574,movie,The Story of the Kelly Gang,1906.0,70,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,1907.0,90,Drama


In [8]:
movies["startYear"] = pd.to_numeric(
    movies["startYear"],
    errors="coerce"
)

In [9]:
movies = movies.dropna(subset=["startYear"])

In [10]:
movies["startYear"] = movies["startYear"].astype(int)

In [11]:
movies["startYear"].describe()

count    641483.000000
mean       1995.197757
std          30.060584
min        1894.000000
25%        1978.000000
50%        2008.000000
75%        2018.000000
max        2032.000000
Name: startYear, dtype: float64

In [12]:
movies = movies[
    (movies["startYear"] >= 1900) &
    (movies["startYear"] <= 2026)
].copy()

print("Movies after year filtering:", len(movies))

Movies after year filtering: 640566


In [13]:
movies["startYear"].describe()

count    640566.000000
mean       1995.155616
std          30.053783
min        1900.000000
25%        1978.000000
50%        2008.000000
75%        2018.000000
max        2026.000000
Name: startYear, dtype: float64

In [14]:
print(movies["startYear"].min())
print(movies["startYear"].max())

1900
2026


In [15]:
movies["runtimeMinutes"] = pd.to_numeric(
    movies["runtimeMinutes"],
    errors="coerce"
)

In [16]:
movies["runtimeMinutes"].describe()

count    468070.000000
mean         89.859519
std         115.443498
min           1.000000
25%          73.000000
50%          89.000000
75%         100.000000
max       51420.000000
Name: runtimeMinutes, dtype: float64

In [20]:
print("Missing genres before cleaning:", movies["genres"].isna().sum())
movies = movies.dropna(subset=["genres"]).copy()

print("Missing genres after cleaning:", movies["genres"].isna().sum())
print("Movies remaining:", len(movies))

Missing genres before cleaning: 0
Missing genres after cleaning: 0
Movies remaining: 569658


In [21]:
movies["genres"].head(20)

570     Action,Adventure,Biography
587                          Drama
610                          Drama
625                          Drama
668                          Drama
672              Adventure,Fantasy
876                          Drama
930                          Drama
1016                        Comedy
1037                     Drama,War
1047                   Documentary
1100                         Drama
1103                         Crime
1135                   Documentary
1151                   Documentary
1163                 Drama,Romance
1172               Adventure,Drama
1228                         Drama
1265                         Drama
1273        Biography,Drama,Family
Name: genres, dtype: object

In [23]:
movies["genre_list"] = movies["genres"].str.split(",")
movies[
    ["primaryTitle", "genres", "genre_list"]
].head(10)

,primaryTitle,genres,genre_list
570,The Story of the Kelly Gang,"Action,Adventure,Biography","[Action, Adventure, Biography]"
587,The Prodigal Son,Drama,[Drama]
610,Robbery Under Arms,Drama,[Drama]
625,Hamlet,Drama,[Drama]
668,Don Quijote,Drama,[Drama]
672,The Fairylogue and Radio-Plays,"Adventure,Fantasy","[Adventure, Fantasy]"
876,"Hamlet, Prince of Denmark",Drama,[Drama]
930,Locura de amor,Drama,[Drama]
1016,Salome Mad,Comedy,[Comedy]
1037,Gøngehøvdingen,"Drama,War","[Drama, War]"


In [1]:
genre_df = movies.explode("genre_list").copy()
genre_df = genre_df.rename(
    columns={"genre_list": "genre"}
)
genre_df[
    ["tconst", "primaryTitle", "startYear", "genre"]
].head(20)

NameError: name 'movies' is not defined